In [1]:
from azureml.core import Workspace, Dataset

from azureml.datadrift import DataDriftDetector

from datetime import datetime, timedelta

In [2]:
ws     = Workspace.from_config()
dstore = ws.get_default_datastore()

In [3]:
dstore.upload('data', 'florida-weather-2019', overwrite=True)

Uploading an estimated of 10 files
Uploading data/2019/01/data.parquet
Uploading data/2019/02/data.parquet
Uploading data/2019/03/data.parquet
Uploading data/2019/04/data.parquet
Uploading data/2019/05/data.parquet
Uploading data/2019/06/data.parquet
Uploading data/2019/07/data.parquet
Uploading data/2019/08/data.parquet
Uploading data/2019/09/data.parquet
Uploading data/2019/10/data.parquet
Uploaded data/2019/10/data.parquet, 1 files out of an estimated total of 10
Uploaded data/2019/04/data.parquet, 2 files out of an estimated total of 10
Uploaded data/2019/08/data.parquet, 3 files out of an estimated total of 10
Uploaded data/2019/01/data.parquet, 4 files out of an estimated total of 10
Uploaded data/2019/09/data.parquet, 5 files out of an estimated total of 10
Uploaded data/2019/05/data.parquet, 6 files out of an estimated total of 10
Uploaded data/2019/02/data.parquet, 7 files out of an estimated total of 10
Uploaded data/2019/06/data.parquet, 8 files out of an estimated total of 

$AZUREML_DATAREFERENCE_05775d55880248cb935643a61a91dc57

In [4]:
dset   = Dataset.Tabular.from_parquet_files(dstore.path('florida-weather-2019/**/data.parquet'))

In [6]:
target = dset.with_timestamp_columns('datetime')
target = target.register(ws, 'target')

In [5]:
target = Dataset.get_by_name(ws, 'target')

In [6]:
baseline = target.time_before(datetime(2019, 2, 1))

In [7]:
features = ['latitude', 'longitude', 'elevation', 'windAngle', 'windSpeed', 'temperature', 'snowDepth', 'stationName', 'countryOrRegion']

In [17]:
monitor = DataDriftDetector.create_from_datasets(ws, 'weatherMonitor', baseline, target, 
                                                      compute_target_name='cpu-cluster', 
                                                      frequency='Week', 
                                                      feature_list=None, 
                                                      drift_threshold=None, 
                                                      latency=0)

In [22]:
monitor = DataDriftDetector.get_by_name(ws, 'drift_monitor2')

monitor = monitor.update(drift_threshold=.6)
monitor = monitor.update(feature_list=features)

In [19]:
backfill1 = monitor.backfill(datetime(2019, 1, 1), datetime(2019, 5, 1))
backfill1

Experiment,Id,Type,Status,Details Page,Docs Page
9ef1ee0e-127d-4c3e-a454-515b0515c845,9ef1ee0e-127d-4c3e-a454-515b0515c845_1570915213060,azureml.scriptrun,Starting,Link to Azure Portal,Link to Documentation


In [20]:
backfill2 = monitor.backfill(datetime(2019, 5, 1), datetime.today())
backfill2

Experiment,Id,Type,Status,Details Page,Docs Page
9ef1ee0e-127d-4c3e-a454-515b0515c845,9ef1ee0e-127d-4c3e-a454-515b0515c845_1570915216625,azureml.scriptrun,Starting,Link to Azure Portal,Link to Documentation


In [13]:
help(DataDriftDetector.backfill)

Help on function backfill in module azureml.datadrift.datadriftdetector:

backfill(self, start_date, end_date, compute_target_name=None, create_compute_target=False)
    Run a backfill job over a given specified start_date and end_date.
    
    *NOTE*: Backfill is only supported on Dataset Based DataDriftDetector objects.
    
    :param start_date:  Date to start the backfill job on
    :type start_date: datetime.datetime
    :param end_date:  End date of backfill job, inclusive
    :type end_date: datetime.datetime
    :param compute_target_name: Optional, AzureML ComputeTarget name, creates one if none is specified
    :type compute_target_name: str
    :param create_compute_target: Optional, whether the DataDriftDetector API should automatically create an AML
                                  compute target. Default to False.
    :type create_compute_target: bool
    :return: DataDriftDetector run
    :rtype: azureml.core.run.Run

